# OOONO API exploration
Run cells from top to bottom using the project `.venv` Python kernel. Fetches **five records per endpoint**, without following pagination. This is schema discovery, not a representative sample or weekly analysis.

Credentials are loaded from the local `.env`; never display the token or credentials. Clear notebook outputs before sharing the submission.

[API documentation](https://qw7ytrkpd3swtndax6c4a73oua0mcrky.lambda-url.eu-west-1.on.aws/docs)

In [ ]:
from pathlib import Path
import os
import pandas as pd
import requests
from dotenv import load_dotenv

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "dbt" / "dbt_project.yml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open the notebook within the project directory.")
if not (ROOT / ".env").exists():
    raise RuntimeError("Open this notebook from the project directory and configure .env.")
load_dotenv(ROOT / ".env")
BASE_URL = "https://qw7ytrkpd3swtndax6c4a73oua0mcrky.lambda-url.eu-west-1.on.aws"
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

## 1. Authenticate
OAuth2 client credentials, sent as form data. Tokens expire after one hour; rerun this cell if necessary. The first API request can take around 20 seconds.

In [ ]:
client_id = os.environ.get("OOONO_CLIENT_ID")
client_secret = os.environ.get("OOONO_CLIENT_SECRET")
if not client_id or not client_secret:
    raise RuntimeError("Set OOONO_CLIENT_ID and OOONO_CLIENT_SECRET in .env.")
response = requests.post(
    f"{BASE_URL}/oauth/token",
    data={"grant_type": "client_credentials", "client_id": client_id, "client_secret": client_secret},
    timeout=(10, 60),
)
if response.status_code != 200:
    raise RuntimeError(f"Authentication failed: HTTP {response.status_code}; response body hidden.")
token_payload = response.json()
session = requests.Session()
session.headers.update({"Authorization": f"Bearer {token_payload['access_token']}"})
del response, token_payload, client_id, client_secret
print("Authenticated; token hidden.")

## 2. Fetch small samples
Preserve each response in `samples` so we can inspect its structure. No full download, pagination loop, or automatic retries.

In [ ]:
SAMPLE_SIZE = 5
samples = {}
frames = {}
for name in ["drives", "warnings", "users"]:
    response = session.get(f"{BASE_URL}/v1/{name}", params={"limit": SAMPLE_SIZE}, timeout=(10, 60))
    if response.status_code != 200:
        raise RuntimeError(f"{name}: HTTP {response.status_code}; response body hidden.")
    payload = response.json()
    samples[name] = payload
    if isinstance(payload, list):
        records = payload
        print(f"{name}: response is a list")
    elif isinstance(payload, dict):
        print(f"{name}: response keys = {list(payload)}")
        list_fields = [key for key, value in payload.items() if isinstance(value, list)]
        if len(list_fields) != 1:
            raise RuntimeError(f"{name}: inspect response structure before selecting the records field.")
        records = payload[list_fields[0]]
        print(f"  records field = {list_fields[0]}, next page available = {bool(payload.get('next_cursor'))}")
    else:
        raise RuntimeError(f"{name}: unexpected response type {type(payload).__name__}")
    frames[name] = pd.json_normalize(records)
    print(f"  sample shape = {frames[name].shape}")
del response

## Drives
One report per drive session, according to the docs.
The first table shows sample rows; the second shows columns, inferred pandas types, and missing values **within this small sample**. Nested JSON objects are flattened using dotted column names.

In [ ]:
drives_df = frames["drives"]
display(drives_df)
display(pd.DataFrame({
    "column": drives_df.columns,
    "inferred_dtype": drives_df.dtypes.astype(str).values,
    "missing_in_sample": drives_df.isna().sum().values,
}))

## Warnings
Warnings and driver reactions.
The first table shows sample rows; the second shows columns, inferred pandas types, and missing values **within this small sample**. Nested JSON objects are flattened using dotted column names.

In [ ]:
warnings_df = frames["warnings"]
display(warnings_df)
display(pd.DataFrame({
    "column": warnings_df.columns,
    "inferred_dtype": warnings_df.dtypes.astype(str).values,
    "missing_in_sample": warnings_df.isna().sum().values,
}))

## Users
Registered users and their current plan; this is current state, not documented plan history.
The first table shows sample rows; the second shows columns, inferred pandas types, and missing values **within this small sample**. Nested JSON objects are flattened using dotted column names.

In [ ]:
users_df = frames["users"]
display(users_df)
display(pd.DataFrame({
    "column": users_df.columns,
    "inferred_dtype": users_df.dtypes.astype(str).values,
    "missing_in_sample": users_df.isna().sum().values,
}))

## Interpretation limits
These are the first five records returned by each endpoint; ordering is not documented. Inferred types and missingness are sample observations, not a schema contract. We still need to investigate duplicates, late arrivals, drive outcomes, version fields, and reporting coverage before defining a bad-drive metric.